
# Brute Force Mask Evaluation — empirical truth table

This notebook evaluates **every possible pruning mask** for the candidate blocks from `cost_loss_table.csv`.

For 10 candidates, it evaluates:

```text
2^10 = 1024 masks
```

The output is one big truth-source table:

```text
brute_force_mask_truth_table.csv
```

Each row means:

```text
one exact mask → real model evaluation → measured loss / accuracy / F1 / compression
```

This notebook is intentionally separate from the QUBO pipeline. The QUBO will later use first-order and second-order data as an approximation, while this brute force table is the empirical benchmark.



## Safety / resume design

This notebook is made for long runs:

```text
evaluate one mask
        ↓
save the row safely to CSV
        ↓
continue with next mask
```

If the run crashes or you stop it, just run the notebook again with `RESUME = True`.

It will:

```text
read brute_force_mask_truth_table.csv
        ↓
find already evaluated masks
        ↓
skip them
        ↓
continue only missing masks
```

It also writes:

```text
brute_force_mask_truth_table.metadata.json
candidate_order_for_brute_force.csv
```


In [ ]:

# =============================
# Configuration
# =============================

MODEL_NAME = "convnext"                 # keep convnext for this project
SPLIT = "test"
MAX_SAMPLES = 600                       # same subset size as current sensitivity table
BATCH_SIZE = 16
NUM_WORKERS = 0

SINGLE_COST_TABLE = "cost_loss_table.csv"
OUTPUT_CSV = "brute_force_mask_truth_table.csv"
CANDIDATE_ORDER_CSV = "candidate_order_for_brute_force.csv"
METADATA_JSON = "brute_force_mask_truth_table.metadata.json"

# Safety / resume
RESUME = True                           # skip masks that are already in OUTPUT_CSV
SAVE_EVERY = 1                          # 1 = safest: save after every mask
BACKUP_EVERY = 50                       # write backup copy every 50 new rows

# Debug settings
# For a quick test, set MAX_MASKS = 3 or 10.
# For the full brute force run, set MAX_MASKS = None.
MAX_MASKS = None

# Optional: skip the empty mask if it already exists. Usually keep False.
SKIP_EMPTY_MASK = False

print("Configuration loaded.")


In [ ]:

# =============================
# Imports
# =============================

from __future__ import annotations

import itertools
import json
import math
import os
import re
import shutil
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import v2
from tqdm.auto import tqdm

import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# =============================
# Image preprocessing and dataset
# Same logic as the existing qnn_and_pruning notebook
# =============================

CLASS_NAMES = [
    "Calgary", "Charlottetown", "Edmonton", "Halifax", "Hamilton",
    "Kitchener-Waterloo", "Montreal", "Ottawa-Gatineau", "Quebec City", "Saskatoon",
    "St Johns", "Toronto", "Vancouver", "Victoria", "Winnipeg",
]


def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    new_img = Image.new("RGB", target_size, (0, 0, 0))
    left = (target_size[0] - img.size[0]) // 2
    top = (target_size[1] - img.size[1]) // 2
    new_img.paste(img, (left, top))
    return new_img


def get_transform(model_name: str):
    if model_name == "convnext":
        return v2.Compose([
            v2.Lambda(lambda img: resize_and_pad(img)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
        ])

    if model_name == "swinv2":
        return transforms.Compose([
            transforms.Resize((192, 192)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ])

    raise ValueError(f"Unknown model_name: {model_name}")


class StreetViewSubset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = list(hf_dataset)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y


def build_dataloader(model_name: str, split: str, max_samples: int, batch_size: int, num_workers: int):
    transform = get_transform(model_name)
    ds = load_dataset(
        "canada-guesser/Canadian-streetview-cities",
        split=f"{split}[:{max_samples}]",
    )
    wrapped = StreetViewSubset(ds, transform)
    return DataLoader(
        wrapped,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


In [ ]:

# =============================
# Model loading and evaluation
# Same logic as existing qnn_and_pruning notebook
# =============================


def load_finetuned_model(model_name: str, device: torch.device):
    if model_name == "convnext":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="cnn_model/convnext_tiny_set_3_final.bin",
        )
        model = timm.create_model("convnext_tiny", pretrained=False, num_classes=15)
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
        model.load_state_dict(state_dict)

    elif model_name == "swinv2":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="vit_model/swinv2_base_window12_192_0_finetuned_canadian_streetview.bin",
        )
        model = timm.create_model("swinv2_base_window12_192", pretrained=False, num_classes=15)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False))

    else:
        raise ValueError(f"Unknown model: {model_name}")

    model.to(device)
    model.eval()
    return model


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, desc: str = "evaluate") -> Dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total = 0
    preds: List[int] = []
    labels: List[int] = []

    for x, y in tqdm(loader, desc=desc, leave=False):
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        if hasattr(logits, "logits"):
            logits = logits.logits
        loss = criterion(logits, y)

        total_loss += float(loss.item())
        total += int(y.numel())
        preds.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        labels.extend(y.detach().cpu().tolist())

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "n_samples": total,
    }


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


In [ ]:

# =============================
# Candidate wrappers
# =============================

@dataclass
class Candidate:
    index: int
    name: str
    module: nn.Module
    n_params: int


class CandidateWrapper(nn.Module):
    """
    Wraps a full model block and allows temporary bypass.

    bypass=False: normal block behavior
    bypass=True : returns input x, simulating block removal while preserving tensor shape
    """

    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module
        self.bypass = False

    def forward(self, x, *args, **kwargs):
        if not self.bypass:
            return self.module(x, *args, **kwargs)
        return x


def _get_parent_and_child(model: nn.Module, module_name: str):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
    child_key = parts[-1]
    return parent, child_key


def replace_module(model: nn.Module, module_name: str, new_module: nn.Module):
    parent, child_key = _get_parent_and_child(model, module_name)
    if child_key.isdigit():
        parent[int(child_key)] = new_module
    else:
        setattr(parent, child_key, new_module)


def wrap_candidates(model: nn.Module, candidate_names: List[str]) -> Tuple[List[Candidate], Dict[str, CandidateWrapper]]:
    module_dict = dict(model.named_modules())
    missing = [name for name in candidate_names if name not in module_dict]
    if missing:
        raise RuntimeError(f"These candidate modules were not found in the model: {missing}")

    candidates: List[Candidate] = []
    wrappers: Dict[str, CandidateWrapper] = {}

    for idx, name in enumerate(candidate_names):
        module = module_dict[name]
        n_params = count_trainable_params(module)
        candidates.append(Candidate(index=idx, name=name, module=module, n_params=n_params))

    # Important: replace after candidate list is built, otherwise named_modules changes during iteration.
    for cand in candidates:
        wrapper = CandidateWrapper(cand.module)
        replace_module(model, cand.name, wrapper)
        wrappers[cand.name] = wrapper

    return candidates, wrappers


def reset_all_bypasses(wrappers: Dict[str, CandidateWrapper]):
    for wrapper in wrappers.values():
        wrapper.bypass = False


def apply_mask(mask: str, candidates: List[Candidate], wrappers: Dict[str, CandidateWrapper]):
    """
    mask bit order follows the row order in cost_loss_table.csv.

    Example for 10 candidates:
        mask = '1000000000' means prune candidate row 0 only.
        mask = '1100000000' means prune candidate row 0 and row 1.
    """
    if len(mask) != len(candidates):
        raise ValueError(f"Mask length {len(mask)} does not match number of candidates {len(candidates)}")

    reset_all_bypasses(wrappers)
    for bit, cand in zip(mask, candidates):
        if bit == "1":
            wrappers[cand.name].bypass = True


def selected_blocks_from_mask(mask: str, candidates: List[Candidate]) -> List[str]:
    return [cand.name for bit, cand in zip(mask, candidates) if bit == "1"]


def selected_params_from_mask(mask: str, candidates: List[Candidate]) -> int:
    return int(sum(cand.n_params for bit, cand in zip(mask, candidates) if bit == "1"))


In [ ]:

# =============================
# Load candidate order from current cost_loss_table.csv
# =============================

single_path = Path(SINGLE_COST_TABLE)
if not single_path.exists():
    raise FileNotFoundError(f"Cannot find {SINGLE_COST_TABLE}. Run this notebook from the project root folder.")

single_df = pd.read_csv(single_path)
if "candidate" not in single_df.columns:
    raise ValueError(f"{SINGLE_COST_TABLE} must contain a 'candidate' column.")

candidate_names = single_df["candidate"].astype(str).tolist()
N = len(candidate_names)
expected_masks = 2 ** N

print(f"Loaded {N} candidates from {SINGLE_COST_TABLE}")
print(f"Brute force will contain 2^{N} = {expected_masks} masks")

pd.DataFrame({
    "candidate_index": list(range(N)),
    "candidate": candidate_names,
    "params_from_single_table": single_df.get("params", pd.Series([np.nan] * N)),
}).to_csv(CANDIDATE_ORDER_CSV, index=False)

print(f"Saved candidate order to {CANDIDATE_ORDER_CSV}")
single_df.head(10)


In [ ]:

# =============================
# Generate all masks
# =============================

all_masks = ["".join(bits) for bits in itertools.product("01", repeat=N)]

if SKIP_EMPTY_MASK:
    all_masks = [m for m in all_masks if "1" in m]

if MAX_MASKS is not None:
    all_masks = all_masks[:MAX_MASKS]

print(f"Masks scheduled in this run: {len(all_masks)}")
print("First masks:", all_masks[:8])
print("Last masks:", all_masks[-8:])


In [ ]:

# =============================
# Safe output helpers
# =============================

OUTPUT_COLUMNS = [
    "mask_index",
    "mask",
    "selected_blocks",
    "num_pruned_blocks",
    "total_params_pruned",
    "candidate_param_reduction_pct",
    "model_param_reduction_pct",
    "baseline_loss",
    "pruned_loss",
    "loss_increase_raw",
    "baseline_accuracy",
    "pruned_accuracy",
    "accuracy_drop_raw",
    "baseline_macro_f1",
    "pruned_macro_f1",
    "f1_drop_raw",
    "n_samples",
    "evaluation_time_sec",
]


def load_existing_results(output_csv: str) -> pd.DataFrame:
    path = Path(output_csv)
    if not path.exists():
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    df = pd.read_csv(path, dtype={"mask": str})
    for col in OUTPUT_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    return df[OUTPUT_COLUMNS]


def atomic_write_csv(df: pd.DataFrame, output_csv: str):
    path = Path(output_csv)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)


def safe_save_result(row: Dict[str, Any], output_csv: str):
    """
    Safe saving method:
    1. Read current CSV.
    2. Replace/add the current mask row.
    3. Write to a temporary file.
    4. Atomically replace the old CSV.

    This is slower than append-only, but much safer for overnight runs.
    With 1024 rows, the overhead is negligible compared to model evaluation time.
    """
    existing = load_existing_results(output_csv)
    existing = existing[existing["mask"].astype(str) != str(row["mask"])]
    updated = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    updated["mask_index"] = pd.to_numeric(updated["mask_index"], errors="coerce")
    updated = updated.sort_values("mask_index", kind="stable").reset_index(drop=True)
    atomic_write_csv(updated[OUTPUT_COLUMNS], output_csv)


def write_metadata(metadata_path: str, metadata: Dict[str, Any]):
    tmp = Path(metadata_path).with_suffix(Path(metadata_path).suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    os.replace(tmp, metadata_path)


def maybe_write_backup(output_csv: str, completed_new: int):
    if completed_new > 0 and completed_new % BACKUP_EVERY == 0 and Path(output_csv).exists():
        backup_name = f"{Path(output_csv).stem}.backup_{completed_new:04d}.csv"
        shutil.copy2(output_csv, backup_name)
        print(f"Backup written: {backup_name}")


existing_df = load_existing_results(OUTPUT_CSV) if RESUME else pd.DataFrame(columns=OUTPUT_COLUMNS)
completed_masks = set(existing_df["mask"].astype(str).tolist())

print(f"Existing completed masks found: {len(completed_masks)}")
if completed_masks:
    print("Example completed masks:", sorted(list(completed_masks))[:5])


In [ ]:

# =============================
# Load model and dataset
# =============================

print("Loading model...")
model = load_finetuned_model(MODEL_NAME, DEVICE)

total_model_trainable_params = count_trainable_params(model)
print(f"Total trainable model parameters: {total_model_trainable_params:,}")

print("Wrapping candidates...")
candidates, wrappers = wrap_candidates(model, candidate_names)

total_candidate_params = sum(c.n_params for c in candidates)
print(f"Total candidate parameters: {total_candidate_params:,}")

candidate_order_df = pd.DataFrame({
    "candidate_index": [c.index for c in candidates],
    "candidate": [c.name for c in candidates],
    "params": [c.n_params for c in candidates],
})
candidate_order_df.to_csv(CANDIDATE_ORDER_CSV, index=False)
print(f"Updated {CANDIDATE_ORDER_CSV} with parameter counts from loaded model.")

display(candidate_order_df)

print("Loading dataset subset...")
loader = build_dataloader(
    model_name=MODEL_NAME,
    split=SPLIT,
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print("Ready.")


In [ ]:

# =============================
# Evaluate baseline once
# =============================

reset_all_bypasses(wrappers)
print("Evaluating baseline model once...")
baseline_start = time.time()
baseline = evaluate(model, loader, DEVICE, desc="baseline")
baseline_time = time.time() - baseline_start

print(
    f"Baseline: loss={baseline['loss']:.6f}, "
    f"acc={baseline['accuracy']:.4f}, "
    f"f1={baseline['macro_f1']:.4f}, "
    f"n={baseline['n_samples']}, "
    f"time={baseline_time:.1f}s"
)


In [ ]:

# =============================
# Brute force evaluation loop
# =============================

existing_df = load_existing_results(OUTPUT_CSV) if RESUME else pd.DataFrame(columns=OUTPUT_COLUMNS)
completed_masks = set(existing_df["mask"].astype(str).tolist())

pending_masks = [m for m in all_masks if (not RESUME or m not in completed_masks)]
print(f"Total masks in this run list: {len(all_masks)}")
print(f"Already completed masks: {len(completed_masks)}")
print(f"Pending masks to evaluate now: {len(pending_masks)}")

metadata = {
    "model_name": MODEL_NAME,
    "split": SPLIT,
    "max_samples": MAX_SAMPLES,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "single_cost_table": SINGLE_COST_TABLE,
    "output_csv": OUTPUT_CSV,
    "candidate_order_csv": CANDIDATE_ORDER_CSV,
    "n_candidates": N,
    "expected_total_masks_full": 2 ** N,
    "masks_scheduled_this_run": len(all_masks),
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "total_model_trainable_params": int(total_model_trainable_params),
    "total_candidate_params": int(total_candidate_params),
    "candidate_order": candidate_names,
    "baseline": baseline,
    "baseline_time_sec": baseline_time,
    "started_or_resumed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
write_metadata(METADATA_JSON, metadata)

newly_completed = 0
run_start = time.time()
eval_times: List[float] = []

try:
    for mask in tqdm(pending_masks, desc="brute force masks"):
        mask_index = int(mask, 2) if mask else 0
        selected_blocks = selected_blocks_from_mask(mask, candidates)
        total_params_pruned = selected_params_from_mask(mask, candidates)

        # Use the already measured baseline for the all-zero mask.
        # This avoids evaluating the exact same baseline twice.
        if set(mask) == {"0"}:
            metrics = baseline
            elapsed = 0.0
        else:
            apply_mask(mask, candidates, wrappers)
            t0 = time.time()
            metrics = evaluate(model, loader, DEVICE, desc=f"mask {mask}")
            elapsed = time.time() - t0
            eval_times.append(elapsed)
            reset_all_bypasses(wrappers)

        row = {
            "mask_index": mask_index,
            "mask": mask,
            "selected_blocks": " | ".join(selected_blocks),
            "num_pruned_blocks": len(selected_blocks),
            "total_params_pruned": int(total_params_pruned),
            "candidate_param_reduction_pct": 100.0 * total_params_pruned / max(total_candidate_params, 1),
            "model_param_reduction_pct": 100.0 * total_params_pruned / max(total_model_trainable_params, 1),
            "baseline_loss": baseline["loss"],
            "pruned_loss": metrics["loss"],
            "loss_increase_raw": metrics["loss"] - baseline["loss"],
            "baseline_accuracy": baseline["accuracy"],
            "pruned_accuracy": metrics["accuracy"],
            "accuracy_drop_raw": baseline["accuracy"] - metrics["accuracy"],
            "baseline_macro_f1": baseline["macro_f1"],
            "pruned_macro_f1": metrics["macro_f1"],
            "f1_drop_raw": baseline["macro_f1"] - metrics["macro_f1"],
            "n_samples": metrics["n_samples"],
            "evaluation_time_sec": elapsed,
        }

        safe_save_result(row, OUTPUT_CSV)
        newly_completed += 1

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        # Update metadata/checkpoint information.
        if newly_completed % SAVE_EVERY == 0:
            avg_eval = float(np.mean(eval_times)) if eval_times else 0.0
            remaining = len(pending_masks) - newly_completed
            eta_sec = avg_eval * remaining
            metadata.update({
                "last_saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "newly_completed_this_run": newly_completed,
                "remaining_this_run": remaining,
                "average_nonzero_mask_eval_time_sec": avg_eval,
                "estimated_remaining_time_sec": eta_sec,
            })
            write_metadata(METADATA_JSON, metadata)

        maybe_write_backup(OUTPUT_CSV, newly_completed)

except KeyboardInterrupt:
    print("Interrupted by user. Progress already saved to CSV. Re-run with RESUME=True to continue.")
    raise

finally:
    reset_all_bypasses(wrappers)
    total_elapsed = time.time() - run_start
    metadata.update({
        "finished_or_interrupted_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_this_run_sec": total_elapsed,
        "newly_completed_this_run": newly_completed,
    })
    write_metadata(METADATA_JSON, metadata)
    print(f"New evaluations completed in this run: {newly_completed}")
    print(f"Output file: {OUTPUT_CSV}")


In [ ]:

# =============================
# Inspect result table
# =============================

result_df = pd.read_csv(OUTPUT_CSV, dtype={"mask": str})
print(f"Rows saved: {len(result_df)}")
print(f"Expected full brute force rows for {N} candidates: {2 ** N}")

display(result_df.head())
display(result_df.tail())


In [ ]:

# =============================
# Quick ranking views
# These do not change the CSV; they are only for inspection.
# =============================

result_df = pd.read_csv(OUTPUT_CSV, dtype={"mask": str})

# Example practical filters. Adjust later depending on your target compression.
TARGET_MODEL_COMPRESSION_PCT = 30.0
MAX_ACCURACY_DROP = None   # example: 0.02 for max 2 percentage points accuracy drop

view = result_df.copy()

# Best masks by lowest validation loss increase, among masks that prune something.
non_empty = view[view["num_pruned_blocks"] > 0].copy()
print("Best masks by lowest loss increase:")
display(non_empty.sort_values(["loss_increase_raw", "model_param_reduction_pct"], ascending=[True, False]).head(10))

print(f"Best masks with model_param_reduction_pct >= {TARGET_MODEL_COMPRESSION_PCT}%:")
filtered = non_empty[non_empty["model_param_reduction_pct"] >= TARGET_MODEL_COMPRESSION_PCT].copy()
if MAX_ACCURACY_DROP is not None:
    filtered = filtered[filtered["accuracy_drop_raw"] <= MAX_ACCURACY_DROP]

display(filtered.sort_values(["loss_increase_raw", "model_param_reduction_pct"], ascending=[True, False]).head(10))



## Output summary

After the full run, the important files are:

```text
brute_force_mask_truth_table.csv
        one row per mask, 1024 rows for 10 candidates

candidate_order_for_brute_force.csv
        defines which candidate corresponds to each bit position

brute_force_mask_truth_table.metadata.json
        configuration, baseline metrics, candidate order, resume/checkpoint information
```

The interpretation is:

```text
cost_loss_table.csv
        first-order single-block evaluations

pair_cost_loss_table.csv
        second-order pair evaluations

brute_force_mask_truth_table.csv
        full empirical truth source for all masks
```
